In [4]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd().parent
JSON_DIR = ROOT / "JSON Whole Model"

json_files = [
    "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json",
    "ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json",
    "ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json",
]

json_paths = [JSON_DIR / name for name in json_files]

# Show clickable file links in the notebook.
links_html = "<br>".join(
    f"<a href='file:///{path.as_posix()}' target='_blank'>{path.name}</a>"
    for path in json_paths
)
display(HTML(f"<b>Selected JSON files:</b><br>{links_html}"))

print("Category summary (duplicate counts) by JSON file:")

file_category_counts = {}
summary_rows = []

for path in json_paths:
    print("\n" + "=" * 80)
    print(f"JSON File: {path.name}")

    if not path.exists():
        missing_df = pd.DataFrame(
            [{"Category": "<missing file>", "Count": 0}]
        )
        display(missing_df)

        file_category_counts[path.name] = pd.Series(dtype="int64")
        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": 0,
                "Total Category Entries": 0,
            }
        )
        continue

    with open(path, "r", encoding="utf-8") as f:
        items = json.load(f)

    categories = []
    for item in items:
        for prop in item.get("Properties", []):
            category = str(prop.get("category", "")).strip()
            if category:
                categories.append(category)

    if categories:
        counts = pd.Series(categories, name="Category").value_counts(dropna=False)
        file_category_counts[path.name] = counts

        file_summary_df = (
            counts
            .rename_axis("Category")
            .reset_index(name="Count")
            .sort_values(["Category"], ascending=[True], kind="stable")
            .reset_index(drop=True)
        )

        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": int(file_summary_df["Category"].nunique()),
                "Total Category Entries": int(file_summary_df["Count"].sum()),
            }
        )
    else:
        file_summary_df = pd.DataFrame(
            [{"Category": "<no categories found>", "Count": 0}]
        )
        file_category_counts[path.name] = pd.Series(dtype="int64")

        summary_rows.append(
            {
                "JSON File": path.name,
                "Distinct Categories": 0,
                "Total Category Entries": 0,
            }
        )

    display(file_summary_df)

print("\n" + "=" * 80)
print("JSON-level summary table:")
json_summary_df = pd.DataFrame(summary_rows)
display(json_summary_df)

print("\n" + "=" * 80)
print("All categories across JSON files ('-' means category not present in that file):")

all_categories = sorted({
    category
    for counts in file_category_counts.values()
    for category in counts.index.tolist()
})

category_matrix_rows = []
for category in all_categories:
    row = {"Category": category}
    for file_name in json_files:
        counts = file_category_counts.get(file_name, pd.Series(dtype="int64"))
        value = counts.get(category)
        row[file_name] = int(value) if pd.notna(value) else "-"
    category_matrix_rows.append(row)

category_matrix_df = pd.DataFrame(category_matrix_rows)
display(category_matrix_df)

Category summary (duplicate counts) by JSON file:

JSON File: ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


,Category,Count
0,ABB4HVDCDesign,168
1,ABB4HVDCDesignRevision,300
2,ABB4ModelRevision Master,52
3,DB Part,52
4,DXF_DWG,4
5,IFC,192
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3



JSON File: ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json


,Category,Count
0,ABB4HVDCDesign,40252
1,ABB4HVDCDesignRevision,53818
2,ABB4ModelRevision Master,3504
3,DB Component Instance,422
4,DB Part,3272
5,IFC,44998
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3



JSON File: ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json


,Category,Count
0,ABB4HVDCDesign,8404
1,ABB4HVDCDesignRevision,17456
2,ABB4ModelRevision Master,1512
3,DB Component Instance,1028
4,DB Part,3072
5,IFC,10463
6,IFCAPPLICATION,3
7,IFCORGANIZATION,1
8,IFCOWNERHISTORY,3
9,IFCPERSON,3



JSON-level summary table:


,JSON File,Distinct Categories,Total Category Entries
0,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,16,3594
1,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,17,567887
2,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json,18,165010



All categories across JSON files ('-' means category not present in that file):


,Category,ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json,ASTIDC-STAN-HE-MPD-RHP1-M-M-0001.json,ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
0,ABB4HVDCDesign,168,40252,8404
1,ABB4HVDCDesignRevision,300,53818,17456
2,ABB4ModelRevision Master,52,3504,1512
3,DB Component Instance,-,422,1028
4,DB Part,52,3272,3072
5,DXF_DWG,4,-,-
6,IFC,192,44998,10463
7,IFCAPPLICATION,3,3,3
8,IFCORGANIZATION,1,1,1
9,IFCOWNERHISTORY,3,3,3
